# TPC-H SF100 SingleStore Benchmark

This notebook sets up an optimized SingleStore TPC-H SF100 environment for the Postgres vs SingleStore webinar demo: https://events.singlestore.com/webinar-postgres-to-singlestore-unlocking-major-performance-enhancements

Goal:
- create a clean optimized schema
- load SF100 TPC-H data from S3
- validate ingest
- run a few representative benchmark queries

Resource: https://www.singlestore.com/spaces/learn-how-to-optimize-performance-with-tpch-100/

In [10]:
%%sql
CREATE DATABASE IF NOT EXISTS s2_tpch_sf10_demo;
USE s2_tpch_sf10_demo;

1 rows affected.

++
||
++
++

## Create optimized tables

These are the optimized SingleStore tables for the webinar baseline. This setup uses unique keys, shard keys, and clustered columnstore indexes.

In [11]:
%%sql
USE s2_tpch_sf10_demo;

CREATE TABLE IF NOT EXISTS `lineitem` (
  `l_orderkey` bigint(11) NOT NULL,
  `l_partkey` int(11) NOT NULL,
  `l_suppkey` int(11) NOT NULL,
  `l_linenumber` int(11) NOT NULL,
  `l_quantity` decimal(15,2) NOT NULL,
  `l_extendedprice` decimal(15,2) NOT NULL,
  `l_discount` decimal(15,2) NOT NULL,
  `l_tax` decimal(15,2) NOT NULL,
  `l_returnflag` char(1) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `l_linestatus` char(1) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `l_shipdate` date NOT NULL,
  `l_commitdate` date NOT NULL,
  `l_receiptdate` date NOT NULL,
  `l_shipinstruct` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `l_shipmode` varchar(10) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `l_comment` varchar(44) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`l_orderkey`,`l_linenumber`) USING HASH,
  SHARD KEY `__SHARDKEY` (`l_orderkey`),
  KEY `l_orderkey` (`l_orderkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `customer` (
  `c_custkey` int(11) NOT NULL,
  `c_name` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `c_address` varchar(40) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `c_nationkey` int(11) NOT NULL,
  `c_phone` varchar(15) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `c_acctbal` decimal(15,2) NOT NULL,
  `c_mktsegment` varchar(10) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `c_comment` varchar(117) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`c_custkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`c_custkey`),
  KEY `c_custkey` (`c_custkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `nation` (
  `n_nationkey` int(11) NOT NULL,
  `n_name` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `n_regionkey` int(11) NOT NULL,
  `n_comment` varchar(152) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`n_nationkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`n_nationkey`),
  KEY `n_nationkey` (`n_nationkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `orders` (
  `o_orderkey` bigint(11) NOT NULL,
  `o_custkey` int(11) NOT NULL,
  `o_orderstatus` char(1) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `o_totalprice` decimal(15,2) NOT NULL,
  `o_orderdate` date NOT NULL,
  `o_orderpriority` varchar(15) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `o_clerk` varchar(15) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `o_shippriority` int(11) NOT NULL,
  `o_comment` varchar(79) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`o_orderkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`o_orderkey`),
  KEY `o_orderkey` (`o_orderkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `part` (
  `p_partkey` int(11) NOT NULL,
  `p_name` varchar(55) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `p_mfgr` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `p_brand` varchar(10) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `p_type` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `p_size` int(11) NOT NULL,
  `p_container` varchar(10) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `p_retailprice` decimal(15,2) NOT NULL,
  `p_comment` varchar(23) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`p_partkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`p_partkey`),
  KEY `p_partkey` (`p_partkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `partsupp` (
  `ps_partkey` int(11) NOT NULL,
  `ps_suppkey` int(11) NOT NULL,
  `ps_availqty` int(11) NOT NULL,
  `ps_supplycost` decimal(15,2) NOT NULL,
  `ps_comment` varchar(199) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`ps_partkey`,`ps_suppkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`ps_partkey`),
  KEY `ps_partkey` (`ps_partkey`,`ps_suppkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `region` (
  `r_regionkey` int(11) NOT NULL,
  `r_name` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `r_comment` varchar(152) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`r_regionkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`r_regionkey`),
  KEY `r_regionkey` (`r_regionkey`) USING CLUSTERED COLUMNSTORE
);

CREATE TABLE IF NOT EXISTS `supplier` (
  `s_suppkey` int(11) NOT NULL,
  `s_name` varchar(25) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `s_address` varchar(40) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `s_nationkey` int(11) NOT NULL,
  `s_phone` varchar(15) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  `s_acctbal` decimal(15,2) NOT NULL,
  `s_comment` varchar(101) CHARACTER SET utf8 COLLATE utf8_bin NOT NULL,
  UNIQUE KEY `pk` (`s_suppkey`) USING HASH,
  SHARD KEY `__SHARDKEY` (`s_suppkey`),
  KEY `s_suppkey` (`s_suppkey`) USING CLUSTERED COLUMNSTORE
);

++
||
++
++

## Create SF100 ingest pipelines

This loads TPC-H SF100 data directly from the public S3 dataset into the optimized tables.

In [16]:
%%sql
USE s2_tpch_sf10_demo;

CREATE OR REPLACE PIPELINE `customer_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/customer'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `customer`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

In [17]:
%%sql
CREATE OR REPLACE PIPELINE `lineitem_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/lineitem/lineitem.'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `lineitem`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

In [28]:
%%sql
CREATE OR REPLACE PIPELINE `nation_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/nation'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `nation`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

In [19]:
%%sql
CREATE OR REPLACE PIPELINE `orders_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/orders'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `orders`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

In [20]:
%%sql
CREATE OR REPLACE PIPELINE `partsupp_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/partsupp'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `partsupp`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

In [41]:
%%sql
CREATE OR REPLACE PIPELINE `part_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/part'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `part`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';


KeyboardInterrupt



In [22]:
%%sql
CREATE OR REPLACE PIPELINE `region_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/region'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `region`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

In [23]:
%%sql
CREATE OR REPLACE PIPELINE `supplier_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_100/supplier'
CONFIG '{"region":"us-east-1", "disable_gunzip": false}'
BATCH_INTERVAL 2500
DISABLE OUT_OF_ORDER OPTIMIZATION
DISABLE OFFSETS METADATA GC
SKIP DUPLICATE KEY ERRORS
INTO TABLE `supplier`
FIELDS TERMINATED BY '|' ENCLOSED BY '' ESCAPED BY '\\'
LINES TERMINATED BY '|\n' STARTING BY '';

++
||
++
++

## Start the pipelines

Run this after the pipeline definitions succeed.

In [30]:
%%sql
USE s2_tpch_sf10_demo;

START PIPELINE customer_pipeline;
START PIPELINE lineitem_pipeline;
START PIPELINE nation_pipeline;
START PIPELINE orders_pipeline;
START PIPELINE partsupp_pipeline;
START PIPELINE part_pipeline;
START PIPELINE region_pipeline;
START PIPELINE supplier_pipeline;

RuntimeError: (singlestoredb.exceptions.OperationalError) 1939: Forwarding Error (node-384c42be-d381-450f-87e5-48c43e93b294-master-0.svc-384c42be-d381-450f-87e5-48c43e93b294:3306): Cannot START PIPELINE because pipeline is currently in the 'Running' State
[SQL: START PIPELINE customer_pipeline;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
If you need help solving this issue, send us a message: https://ploomber.io/community


## Check pipeline status

This helps confirm whether the load completed successfully.

In [36]:
%%sql
SELECT
  CONCAT(PIPELINE_NAME) AS pipeline_id,
  sub.BATCH_STATE AS last_batch_state,
  IFNULL(sub.BATCH_ROWS_WRITTEN, 0) AS last_batch_rows_written
FROM (
  SELECT
    DATABASE_NAME,
    PIPELINE_NAME,
    BATCH_STATE,
    BATCH_ROWS_WRITTEN,
    ROW_NUMBER() OVER (PARTITION BY DATABASE_NAME, PIPELINE_NAME) AS r
  FROM INFORMATION_SCHEMA.PIPELINES_BATCHES_METADATA
  WHERE BATCH_STATE NOT IN ('No Data', 'In Progress')
) sub
WHERE r = 1
  AND DATABASE_NAME = 's2_tpch_sf10_demo'
ORDER BY pipeline_id ASC;

8 rows affected.

pipeline_id,last_batch_state,last_batch_rows_written
customer_pipeline,Succeeded,15000000
lineitem_pipeline,Succeeded,27637902
nation_pipeline,Succeeded,25
orders_pipeline,Succeeded,28800000
partsupp_pipeline,Queued,0
part_pipeline,Queued,0
region_pipeline,Succeeded,5
supplier_pipeline,Succeeded,1000000


## Sanity-check row counts

If these are non-zero across all tables, ingest worked.

In [40]:
%%sql
SELECT 'customer' AS table_name, COUNT(*) AS row_count FROM customer
UNION ALL
SELECT 'lineitem', COUNT(*) FROM lineitem
UNION ALL
SELECT 'nation', COUNT(*) FROM nation
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'part', COUNT(*) FROM part
UNION ALL
SELECT 'partsupp', COUNT(*) FROM partsupp
UNION ALL
SELECT 'region', COUNT(*) FROM region
UNION ALL
SELECT 'supplier', COUNT(*) FROM supplier;

8 rows affected.

table_name,row_count
customer,15000000
lineitem,56437902
nation,25
orders,57600000
part,0
partsupp,57600000
region,5
supplier,1000000


## Benchmark query 1

This is one of the representative TPC-H queries from the optimization notebook.

In [ ]:
%%sql
USE s2_tpch_sf10_demo;

SELECT
    l_returnflag,
    l_linestatus,
    SUM(l_quantity) AS sum_qty,
    SUM(l_extendedprice) AS sum_base_price,
    SUM(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
    SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
    AVG(l_quantity) AS avg_qty,
    AVG(l_extendedprice) AS avg_price,
    AVG(l_discount) AS avg_disc,
    COUNT(*) AS count_order
FROM lineitem
WHERE l_shipdate <= DATE('1998-12-01') - INTERVAL '90' DAY
GROUP BY l_returnflag, l_linestatus
ORDER BY l_returnflag, l_linestatus;

## Benchmark query 4

In [ ]:
%%sql
USE s2_tpch_sf10_demo;

SELECT
    o_orderpriority,
    COUNT(*) AS order_count
FROM orders
WHERE o_orderdate >= DATE('1993-07-01')
  AND o_orderdate < DATE('1993-10-01')
  AND EXISTS (
      SELECT *
      FROM lineitem
      WHERE l_orderkey = o_orderkey
        AND l_commitdate < l_receiptdate
  )
GROUP BY o_orderpriority
ORDER BY o_orderpriority;

## Benchmark query 21

In [ ]:
%%sql
USE s2_tpch_sf10_demo;

SELECT
    s_name,
    COUNT(*) AS numwait
FROM supplier,
     lineitem l1,
     orders,
     nation
WHERE s_suppkey = l1.l_suppkey
  AND o_orderkey = l1.l_orderkey
  AND o_orderstatus = 'F'
  AND l1.l_receiptdate > l1.l_commitdate
  AND EXISTS (
      SELECT *
      FROM lineitem l2
      WHERE l2.l_orderkey = l1.l_orderkey
        AND l2.l_suppkey <> l1.l_suppkey
  )
  AND NOT EXISTS (
      SELECT *
      FROM lineitem l3
      WHERE l3.l_orderkey = l1.l_orderkey
        AND l3.l_suppkey <> l1.l_suppkey
        AND l3.l_receiptdate > l3.l_commitdate
  )
  AND s_nationkey = n_nationkey
  AND n_name = 'EGYPT'
GROUP BY s_name
ORDER BY numwait DESC, s_name
LIMIT 100;

## Alternative lineitem pipeline (if needed)

If the lineitem pipeline encounters errors, you can try this alternate configuration.

In [ ]:
%%sql
CREATE OR REPLACE PIPELINE `lineitem_pipeline`
AS LOAD DATA S3 'memsql-tpch-dataset/sf_10/lineitem/'
CONFIG '{"region":"us-east-1"}'
SKIP DUPLICATE KEY ERRORS
INTO TABLE lineitem
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '|\n';